# 01a. Data Integration: Unified Dataset Assembly

This notebook executes the integration of the manual historical dataset with the SerpApi response records. By merging these two distinct data sources and performing subsequent enrichment, a unified, feature-rich master dataset was created, that provides the necessary depth for our predictive model.

## Objectives
- Dataset Consolidation: Merge the manually captured records with the processed SerpApi batch output into a single, cohesive master DataFrame.
- Feature Enrichment (API-Specific): Generate the is_lcc (Low-Cost Carrier) classification for all API-fetched records to ensure feature parity with the manual dataset.
- External Variable Injection: Apply batch enrichment for demand-driven indicators: is_holiday_sin, is_holiday_other, and is_sch_holiday.
- Consistency Check: Verify schema uniformity across the combined dataset to ensure zero data loss or type mismatch during the append process.

### Environment Setup

In [1]:
import numpy as np
import pandas as pd

!pip install holidays
import holidays

from datetime import timedelta, datetime

In [2]:
# Load the manual dataset
df_manual = pd.read_csv("manualdataset.csv")

# Load API datasets
df_api = pd.read_csv('./serpapi_response.csv')

# Load LCC list
df_lcc = pd.read_excel('./LCC List.xlsx')

### Carrier Type

In [3]:
# Map carrier_type from LCC List. If airline_code exists in LCC List, carrier_code = LCC, otherwise FSC. 
# Identify the column to use in LCC List
lcc_col_name = 'airline_code' if 'airline_code' in df_lcc.columns else df_lcc.columns[1]

# Transform CSV column to a lookup set. Then perform text normalisation (strip spaces, and convert to uppercase)
lcc_set = set(df_lcc[lcc_col_name].astype(str).str.strip().str.upper())
print(f"Found {len(lcc_set)} valid LCC codes for mapping index")

# Lookup carrier to LCC List
df_api['is_lcc'] = np.where(df_api['airline_code'].isin(lcc_set),True,False)

Found 3 valid LCC codes for mapping index


### Combining Manual and API datasets

In [4]:
# Combine manual and api datasets
df = pd.concat([df_manual, df_api], ignore_index=True)

### Adding Features

#### Public Holidays Feature

**+/- 2 Day Holiday Buffer**  
Travel demand is rarely contained within a single calendar date. By capturing the two days before and after a public holiday, we are creating a "Travel Event Window."

- Anticipatory Demand: Travelers often shift their departure date to the Friday evening or Saturday morning to maximize their leave days.
- The "Return Spike": The two days following a holiday (e.g., a Sunday or Monday return) are often the highest-priced days of the entire event because all holiday-makers are trying to return simultaneously.
- Modeling Benefit: By including these buffers, the regression model learns the "shoulder" effects—the transition period where prices ramp up and scale down.

In [5]:
def is_holiday_feature(df, buffer_days =2):
    df['departure_date'] = pd.to_datetime(df['departure_date'])
    start_year = df['departure_date'].min().year
    end_year = df['departure_date'].max().year 
    years_range = list(range(start_year, end_year +1))

    airport_to_country = {
        'LHR': 'GB',
        'NRT': 'JP',
        'BKK': 'TH',
        'MEL': 'AU',
        'SIN': 'SG'
    }

    # Create Sets of dates in memory
    def holiday_set(country_iso):
        country_hols = holidays.country_holidays(country_iso, years=years_range)
        # Set is used here to prevent duplicates
        expanded_set = set()

        for h_date in country_hols.keys():
            # To capture 2 days before and after a public holiday. We assume travellers will maximise the public holidays for a longer travel period.
            core_window = [h_date + timedelta(days=i) for i in range(-buffer_days, buffer_days+1)]

            # Adding dates from the core window into the expanded set
            for date_item in core_window:
                expanded_set.add(date_item)
                # Finds the weekday of the dates in the core window
                weekday = date_item.weekday()

                # If the public holiday falls on a Saturday, we want to capture Friday too as we assume travellers might depart ealier.     
                if weekday ==5:
                    expanded_set.add(date_item - timedelta(days=1))

                # If the public holiday falls on a Sunday, we want to capture Monday for people flying back home. 
                elif weekday ==6:
                    expanded_set.add(date_item + timedelta(days=1))
        return expanded_set

    # Generate lookup sets
    # Fetch all Singapore holidays
    sg_bridged_set = holiday_set('SG')

    df['is_holiday_sin'] = df['departure_date'].dt.date.isin(sg_bridged_set).astype(int)

    # Fetch for all other destinations
    dest_maps = {}
    for code, country_iso in airport_to_country.items():
        if code == 'SIN': continue
        dest_maps[code] = holiday_set(country_iso)

    def dest_holiday(row):
        dest = row['other_airport']
        if dest not in dest_maps:
            return 0
        return 1 if row['departure_date'].date() in dest_maps[dest] else 0

    df['is_holiday_other'] = df.apply(dest_holiday, axis=1)
    return df

# Ending public holidays job
df = is_holiday_feature(df)
print(f"Public holidays feature completed.")

Public holidays feature completed.


#### (Singapore) School Holidays Feature

**Friday before the 1st**  
Capturing the Friday before 1 June or 1 December aims to capture the School Holiday Transition.

- Operational Reality: 1 June or 1 December often marks the beginning of the school holidays in Singapore. Families with school-aged children are highly price-sensitive but time-constrained; they will almost exclusively depart on the last day of the school term (the Friday) to gain an extra two days of vacation.
- The Price "Step-Up": This Friday represents the exact moment the pricing "fare bucket" shifts from a "normal" base rate to a "holiday/peak" rate.

In [6]:
def add_sch_holidays(df):
    def check_school_holiday(row):
        flight_date = row['departure_date'].date()
        current_year = flight_date.year
        month = flight_date.month

        # Check if departure date is in Jun or Dec
        if month in [6,12]:
            return 1

        # If 1 Jun or 1 Dec falls on a weekend, make Friday considered a school holiday.
        # No extension of window for 30 Jun/31 Dec as children need to attend school. 
        # For 1 Jun
        jun1 = datetime(current_year, 6, 1).date()
        jun1_weekday = jun1.weekday() # 5 = Sat; 6 = Sun
        if jun1_weekday == 5 and flight_date == (jun1 - timedelta(days=1)):
            return 1
        if jun1_weekday == 6 and flight_date == (jun1 - timedelta(days=2)):
            return 1

        # For 1 Dec 
        dec1 = datetime(current_year, 12, 1).date()
        dec1_weekday = dec1.weekday() # 5 = Sat; 6 = Sun
        if dec1_weekday == 5 and flight_date == (dec1 - timedelta(days=1)):
            return 1
        if dec1_weekday == 6 and flight_date == (dec1 - timedelta(days=2)):
            return 1

        return 0

    df['is_sch_holiday'] = df.apply(check_school_holiday, axis=1)
    return df

df = add_sch_holidays(df)
print(f"School holidays feature completed.")

School holidays feature completed.


### Data Export

This combined output is saved as dataset_1bcombine.csv to ensure clear data lineage and to prevent accidental overwriting of the master dataset.csv

In [ ]:
# Combine manual and api datasets
df.to_csv('1b. dataset.csv', index=False)